# Smart Container Risk Detection Engine
## Hybrid Ensemble — XGBoost + Random Forest + Isolation Forest

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from xgboost import XGBClassifier

## 1 · Load Data

In [ ]:
COL_RENAME = {
    "Declaration_Date (YYYY-MM-DD)": "Declaration_Date",
    "Trade_Regime (Import / Export / Transit)": "Trade_Regime",
}

def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).rename(columns=COL_RENAME)
    return df

df_hist = load_data("Historical Data.csv")
df_rt   = load_data("Real-Time Data.csv")

## 2 · Preprocessing

In [ ]:
RISK_STATUSES = {"critical", "hold", "inspect", "detained", "flagged"}
NUM_COLS = ["Declared_Value", "Declared_Weight", "Measured_Weight", "Dwell_Time_Hours"]
CAT_COLS = ["Trade_Regime", "Origin_Country", "Destination_Country",
            "Destination_Port", "Shipping_Line"]

def preprocess(df: pd.DataFrame, has_label: bool = True) -> pd.DataFrame:
    df = df.copy()
    df["Declaration_Date"] = pd.to_datetime(df["Declaration_Date"], errors="coerce")
    df["declaration_month"] = df["Declaration_Date"].dt.month.fillna(1).astype(int)
    df["declaration_dow"]   = df["Declaration_Date"].dt.dayofweek.fillna(0).astype(int)

    for col in NUM_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col].fillna(df[col].median(), inplace=True)
    for col in ["Declared_Value", "Declared_Weight"]:
        df[col] = df[col].clip(upper=df[col].quantile(0.99))
    for col in CAT_COLS:
        df[col].fillna(df[col].mode()[0], inplace=True)

    if has_label and "Clearance_Status" in df.columns:
        df["Risk_Label"] = (
            df["Clearance_Status"].str.strip().str.lower()
            .isin(RISK_STATUSES).astype(int)
        )
    return df

df_hist = preprocess(df_hist, has_label=True)
df_rt   = preprocess(df_rt,   has_label=False)
print(f"Historical: {df_hist.shape} | Target: {df_hist['Risk_Label'].value_counts().to_dict()}")

## 3 · Feature Engineering
Only physically-meaningful shipment features — no entity risk rates to avoid leakage.

In [ ]:
EPS = 1e-5

def engineer_features(df: pd.DataFrame, train_stats: dict = None):
    """Build domain features. Compute stats from train if train_stats is None."""
    df = df.copy()

    # ── Weight anomaly features ──
    df["weight_diff"]          = df["Measured_Weight"] - df["Declared_Weight"]
    df["weight_ratio"]         = df["Measured_Weight"] / (df["Declared_Weight"] + EPS)
    df["weight_deviation_pct"] = (df["weight_diff"].abs() / (df["Declared_Weight"] + EPS)).clip(upper=0.5)
    df["value_per_weight"]     = df["Declared_Value"] / (df["Declared_Weight"] + EPS)

    # ── Commodity weight benchmark (leak-free) ──
    if train_stats is None:
        train_stats = {}
        hs_grp = df.groupby("HS_Code")["Declared_Weight"]
        train_stats["hs_avg"] = hs_grp.mean()
        train_stats["hs_std"] = hs_grp.std().fillna(1)
        train_stats["hs_global_avg"] = df["Declared_Weight"].mean()
        train_stats["hs_global_std"] = max(df["Declared_Weight"].std(), 1)
        train_stats["dwell_mean"] = df["Dwell_Time_Hours"].mean()
        train_stats["dwell_std"]  = max(df["Dwell_Time_Hours"].std(), 1)

    df["commodity_avg_weight"] = df["HS_Code"].map(train_stats["hs_avg"]).fillna(train_stats["hs_global_avg"])
    df["commodity_std_weight"] = df["HS_Code"].map(train_stats["hs_std"]).fillna(train_stats["hs_global_std"])
    df["commodity_weight_zscore"] = (
        (df["Declared_Weight"] - df["commodity_avg_weight"]) / (df["commodity_std_weight"] + EPS)
    ).clip(-3, 3)

    # ── Dwell time z-score ──
    df["dwell_time_zscore"] = (
        (df["Dwell_Time_Hours"] - train_stats["dwell_mean"]) / (train_stats["dwell_std"] + EPS)
    ).clip(-3, 3)

    return df, train_stats

df_hist, _ = engineer_features(df_hist)
print(f"Engineered: {df_hist.shape}")

## 4 · Encode & Stratified Split

In [ ]:
DROP_COLS = [
    "Container_ID", "Declaration_Date", "Declaration_Time",
    "Clearance_Status", "Importer_ID", "Exporter_ID", "HS_Code",
    "Risk_Label", "commodity_avg_weight", "commodity_std_weight",
]

FEATURE_COLS = [
    "Trade_Regime", "Origin_Country", "Destination_Country",
    "Destination_Port", "Shipping_Line",
    "Declared_Value", "Declared_Weight", "Measured_Weight",
    "Dwell_Time_Hours", "declaration_month", "declaration_dow",
    "weight_diff", "weight_ratio", "weight_deviation_pct",
    "value_per_weight", "commodity_weight_zscore", "dwell_time_zscore",
]

def encode(df, fit=True, encoders=None):
    df = df.copy()
    if encoders is None:
        encoders = {}
    for col in CAT_COLS:
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        else:
            le = encoders[col]
            known = set(le.classes_)
            df[col] = df[col].astype(str).apply(lambda v: v if v in known else le.classes_[0])
            df[col] = le.transform(df[col])
    avail = [c for c in FEATURE_COLS if c in df.columns]
    X = df[avail]
    y = df["Risk_Label"].astype(int) if "Risk_Label" in df.columns else None
    return X, y, encoders

# Full encode to fit encoders
_, _, encoders = encode(df_hist, fit=True)

# Stratified split FIRST, then re-engineer with train-only stats
X_idx = df_hist.index.values
y_all = df_hist["Risk_Label"].values
tr_idx, te_idx = train_test_split(X_idx, test_size=0.2, stratify=y_all, random_state=42)

df_train_raw = df_hist.loc[tr_idx].copy()
df_test_raw  = df_hist.loc[te_idx].copy()

# Re-engineer with TRAIN-ONLY statistics
df_train_eng, train_stats = engineer_features(preprocess(load_data("Historical Data.csv")).loc[tr_idx], train_stats=None)
df_test_eng, _ = engineer_features(preprocess(load_data("Historical Data.csv")).loc[te_idx], train_stats=train_stats)

X_train, y_train, _ = encode(df_train_eng, fit=False, encoders=encoders)
X_test,  y_test,  _ = encode(df_test_eng,  fit=False, encoders=encoders)

# Standardize numeric features
scaler = StandardScaler()
num_feats = [c for c in X_train.columns if c not in CAT_COLS]
X_train[num_feats] = scaler.fit_transform(X_train[num_feats])
X_test[num_feats]  = scaler.transform(X_test[num_feats])

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train target: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Features ({len(FEATURE_COLS)}): {list(X_train.columns)}")

## 5 · Model Training — Heavy Regularization

In [ ]:
# Isolation Forest feature subset (non-weight features to avoid dominance)
IF_FEATURES = ["value_per_weight", "Dwell_Time_Hours", "dwell_time_zscore"]

# ── XGBoost — heavily regularized ──
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / max(pos, 1)

xgb_model = XGBClassifier(
    n_estimators=80,
    max_depth=2,
    learning_rate=0.05,
    subsample=0.5,
    colsample_bytree=0.4,
    min_child_weight=50,
    gamma=5.0,
    reg_alpha=5.0,
    reg_lambda=10.0,
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)

# ── Random Forest — very constrained ──
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=50,
    min_samples_split=100,
    max_features=0.3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

# ── Isolation Forest ──
iso_model = IsolationForest(
    n_estimators=100,
    max_samples=0.5,
    contamination=0.02,
    random_state=42,
    n_jobs=-1,
)
iso_model.fit(X_train[IF_FEATURES])

print("Models trained successfully.")

## 6 · Ensemble Risk Scoring

In [ ]:
W_XGB, W_RF, W_IF = 0.50, 0.30, 0.20

def if_anomaly_score(model, X_if, train_raw=None):
    raw = -model.decision_function(X_if)
    if train_raw is not None:
        lo, hi = train_raw.min(), train_raw.max()
    else:
        lo, hi = raw.min(), raw.max()
    return np.clip((raw - lo) / (hi - lo + 1e-9), 0, 1)

def ensemble_score(xgb, rf, iso, X, X_if, train_if_raw=None):
    xgb_p = xgb.predict_proba(X)[:, 1]
    rf_p  = rf.predict_proba(X)[:, 1]
    if_s  = if_anomaly_score(iso, X_if, train_if_raw)
    return np.round((W_XGB * xgb_p + W_RF * rf_p + W_IF * if_s) * 100, 2)

# Cache train IF baseline
train_if_raw = -iso_model.decision_function(X_train[IF_FEATURES])

train_scores = ensemble_score(xgb_model, rf_model, iso_model, X_train, X_train[IF_FEATURES], train_if_raw)
test_scores  = ensemble_score(xgb_model, rf_model, iso_model, X_test,  X_test[IF_FEATURES],  train_if_raw)

## 7 · Risk Labels & Evaluation

In [ ]:
def risk_label(score):
    if score >= 70: return "High Risk"
    if score >= 30: return "Medium Risk"
    return "Low Risk"

def evaluate(tag, y_true, scores, threshold=50):
    y_pred = (scores >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="weighted")
    auc = roc_auc_score(y_true, scores / 100)
    print(f"  {tag:6s} | Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    return acc, f1, auc

print("=" * 60)
print("  Ensemble Evaluation")
print("=" * 60)
tr_a, tr_f, tr_auc = evaluate("Train", y_train, train_scores)
te_a, te_f, te_auc = evaluate("Test ", y_test,  test_scores)
print(f"\n  Overfit gap → Acc: {tr_a-te_a:+.4f} | F1: {tr_f-te_f:+.4f} | AUC: {tr_auc-te_auc:+.4f}")
print(f"\n  Test risk dist: {pd.Series([risk_label(s) for s in test_scores]).value_counts().to_dict()}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"  5-Fold CV AUC (XGB only): {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

## 8 · Explanation Generation

In [ ]:
FEAT_DESC = {
    "weight_diff":             "weight mismatch between declared and measured",
    "weight_ratio":            "abnormal measured-to-declared weight ratio",
    "weight_deviation_pct":    "significant weight deviation detected",
    "value_per_weight":        "unusual value-to-weight ratio",
    "commodity_weight_zscore": "weight unusual for this commodity type",
    "dwell_time_zscore":       "abnormal dwell time in port",
    "Dwell_Time_Hours":        "extended port dwell time",
    "Declared_Value":          "unusual declared cargo value",
}

def generate_explanations(xgb_model, X, scores):
    imp = xgb_model.feature_importances_
    feat_names = X.columns.tolist()
    top_feats = [feat_names[i] for i in np.argsort(imp)[::-1][:5]]

    explanations = []
    for i in range(len(X)):
        s = scores[i]
        if s < 30:
            explanations.append("No significant risk signals detected.")
            continue
        row = X.iloc[i]
        signals = []
        for f in top_feats:
            desc = FEAT_DESC.get(f, f.replace("_", " "))
            v = row[f]
            if f in ("commodity_weight_zscore", "dwell_time_zscore"):
                if abs(v) > 1.0: signals.append(desc)
            elif f == "weight_deviation_pct":
                if v > 0.08: signals.append(desc)
            else:
                signals.append(desc)
            if len(signals) >= 3: break

        if not signals:
            explanations.append("Ensemble model detected elevated risk pattern." if s >= 70
                                else "Moderate risk signals detected by ensemble model.")
        else:
            explanations.append("; ".join(signals).capitalize() + ".")
    return explanations

## 9 · Real-Time Inference & Export

In [ ]:
def run_inference(df_new, xgb_model, rf_model, iso_model,
                  encoders, train_stats, scaler, num_feats, train_if_raw):
    df_eng, _ = engineer_features(df_new, train_stats=train_stats)
    X_new, _, _ = encode(df_eng, fit=False, encoders=encoders)
    X_new[num_feats] = scaler.transform(X_new[num_feats])

    scores = ensemble_score(xgb_model, rf_model, iso_model,
                            X_new, X_new[IF_FEATURES], train_if_raw)
    return pd.DataFrame({
        "Container_ID":  df_new["Container_ID"].values,
        "Risk_Score":    scores,
        "Risk_Label":    [risk_label(s) for s in scores],
        "Explanation":   generate_explanations(xgb_model, X_new, scores),
    })

# ── Historical predictions ──
df_hist_eng, _ = engineer_features(df_hist, train_stats=train_stats)
X_hist, _, _ = encode(df_hist_eng, fit=False, encoders=encoders)
X_hist[num_feats] = scaler.transform(X_hist[num_feats])
hist_scores = ensemble_score(xgb_model, rf_model, iso_model,
                             X_hist, X_hist[IF_FEATURES], train_if_raw)
results_hist = pd.DataFrame({
    "Container_ID": df_hist["Container_ID"].values,
    "Risk_Score":   hist_scores,
    "Risk_Label":   [risk_label(s) for s in hist_scores],
    "Explanation":  generate_explanations(xgb_model, X_hist, hist_scores),
})
results_hist.to_csv("risk_predictions.csv", index=False)
print(f"Historical: {len(results_hist):,} rows → risk_predictions.csv")
print(f"  {results_hist['Risk_Label'].value_counts().to_dict()}")

# ── Real-time predictions ──
results_rt = run_inference(df_rt, xgb_model, rf_model, iso_model,
                           encoders, train_stats, scaler, num_feats, train_if_raw)
results_rt.to_csv("risk_predictions_realtime.csv", index=False)
print(f"\nReal-Time: {len(results_rt):,} rows → risk_predictions_realtime.csv")
print(f"  {results_rt['Risk_Label'].value_counts().to_dict()}")

results_rt.head(10)